# Name That LLM — production submission

This notebook trains the deployable no-context model on the complete competition training set and writes `submission.parquet`.

**Model:** word TF-IDF + character TF-IDF + punctuation/style features + class-balanced LinearSVC.

The test file contains only `id` and `text`, so this pipeline does not require `prompt_id`, same-prompt peers, internet access, or external data.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
OUTPUT_PATH = Path('/kaggle/working/submission.parquet')

VALID_LABELS = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3',
]

WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGE = (1, 5)
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 2.0
MAX_ITER = 2_000
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

assert TRAIN_PATH.exists(), f'Missing training file: {TRAIN_PATH}'
assert TEST_PATH.exists(), f'Missing test file: {TEST_PATH}'
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
assert {'text', 'model'}.issubset(train.columns), 'Training file must contain text and model columns'
assert {'id', 'text'}.issubset(test.columns), 'Test file must contain id and text columns'
assert train['model'].isin(VALID_LABELS).all(), 'Unexpected label in training data'

train_text = train['text'].fillna('').astype(str).to_numpy()
test_text = test['text'].fillna('').astype(str).to_numpy()
y = train['model'].astype(str).to_numpy()
print(f'Train rows: {len(train):,}; test rows: {len(test):,}; classes: {len(np.unique(y))}')


In [ ]:
def punctuation_features(items):
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def style_features(items):
    rows = []
    for text in items:
        words = re.findall(r"\b[\w']+\b", text)
        rows.append([
            float(len(text)), float(len(words)),
            float(len(re.findall(r'[.!?]+', text))),
            float(sum(text.count(c) for c in PUNCTUATION)),
            float(sum(c.isdigit() for c in text)),
            float(sum(c.isupper() for c in text)),
            float(text.count('?')), float(text.count('!')),
            float(text.count(':')), float(text.count(';')),
            float(sum(text.count(c) for c in '()[]{}')),
            float(sum(text.count(c) for c in '#*_`')),
            float(sum(text.count(c) for c in '*-•')),
            float(text.count('\n')),
        ])
    return np.asarray(rows, dtype=np.float32)

word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=WORD_NGRAM_RANGE, min_df=MIN_DF,
    sublinear_tf=True, max_features=MAX_FEATURES,
)
char_vectorizer = TfidfVectorizer(
    analyzer='char', ngram_range=CHAR_NGRAM_RANGE, min_df=MIN_DF,
    sublinear_tf=True, max_features=MAX_FEATURES,
)

X_word_train = word_vectorizer.fit_transform(train_text)
X_word_test = word_vectorizer.transform(test_text)
X_char_train = char_vectorizer.fit_transform(train_text)
X_char_test = char_vectorizer.transform(test_text)

punct_scaler = StandardScaler()
X_punct_train = csr_matrix(punct_scaler.fit_transform(punctuation_features(train_text)))
X_punct_test = csr_matrix(punct_scaler.transform(punctuation_features(test_text)))

style_scaler = StandardScaler()
X_style_train = csr_matrix(style_scaler.fit_transform(style_features(train_text)))
X_style_test = csr_matrix(style_scaler.transform(style_features(test_text)))

X_train = hstack([X_word_train, X_char_train, X_punct_train, X_style_train], format='csr')
X_test = hstack([X_word_test, X_char_test, X_punct_test, X_style_test], format='csr')
print(f'Feature matrix: train={X_train.shape}; test={X_test.shape}')


In [ ]:
model = LinearSVC(C=SVC_C, class_weight='balanced', max_iter=MAX_ITER)
model.fit(X_train, y)
predictions = model.predict(X_test)

assert len(predictions) == len(test), 'Prediction count does not match test rows'
assert set(predictions).issubset(set(VALID_LABELS)), 'Prediction contains an invalid model label'
assert test['id'].notna().all(), 'Test contains null ids'
assert test['id'].is_unique, 'Test ids are not unique'

submission = pd.DataFrame({'id': test['id'].to_numpy(), 'model': predictions})
assert list(submission.columns) == ['id', 'model']
assert submission['id'].is_unique and submission['id'].notna().all()
assert submission['model'].notna().all()
submission.to_parquet(OUTPUT_PATH, index=False)

print(f'Wrote: {OUTPUT_PATH}')
print(f'Submission rows: {len(submission):,}')
print('Prediction distribution:')
print(submission['model'].value_counts().sort_index().to_string())
print('Submission validation: PASSED')


## Submission checklist

The output is `submission.parquet` with exactly two columns: `id` and `model`. Every test id appears once, no values are null, and every prediction belongs to the official 18-label set.
